Importing necessary libraries

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
from PIL import Image
import random

Choosing which dataset is used

In [9]:
root = "Poles2025"
dataset = "roadpoles_v1" 
dataset_path = Path(root) / dataset

Checking number of train/val/test images

In [10]:
split_counts = {}

for split in ["train", "valid", "test"]:
    image_dir = dataset_path / split / "images"
    images = list(image_dir.glob("*.PNG"))
    split_counts[split] = len(images)
    
# print(split_counts)
    
print(f"Training set: {split_counts["train"]} images")
print(f"Validation set: {split_counts["valid"]} images")
print(f"Testing set: {split_counts["test"]} images")



Training set: 322 images
Validation set: 92 images
Testing set: 46 images


Checking number of annotations per split, negative samples and storing all cx, cy, w, and h values in lists. 

In [11]:
negative_samples = 0
all_cx, all_cy, all_w, all_h = [], [], [], []
boxes_per_image = {"train": [], "valid": []}

for split in ["train", "valid"]:
    label_dir = dataset_path / split / "labels"
    labels = list(label_dir.glob("*.txt"))
    
    total_annotations = 0
    
    for label_file in labels:
        with open(label_file) as file:
            lines = file.readlines()
            total_annotations += len(lines)
            boxes_per_image[split].append(len(lines))
            
            if len(lines) == 0:
                negative_samples += 1
            
            for line in lines:
                _, cx, cy, w, h = line.split()
                all_cx.append(float(cx))
                all_cy.append(float(cy))
                all_w.append(float(w))
                all_h.append(float(h))
    
    avg = total_annotations / len(labels) if labels else 0
    print(f"{split}: {total_annotations} annotations, {avg:.2f} per image")

print(f"\nTotal bounding boxes: {len(all_cx)}")
print(f"Negative samples (no poles): {negative_samples}")


train: 392 annotations, 1.22 per image
valid: 113 annotations, 1.23 per image

Total bounding boxes: 505
Negative samples (no poles): 0


Total boxes per image for train

In [12]:
fig = px.histogram(
    x=boxes_per_image["train"] + boxes_per_image["valid"],
    color=["train"] * len(boxes_per_image["train"]) + ["valid"] * len(boxes_per_image["valid"]),
    barmode="overlay",
    nbins=10,
    title="Total boxes per image (train vs val)",
    labels={"x": "Total boxes", "color": "Split"}
)

fig.update_xaxes(tickvals=[1,2,3])

fig.show()

Scatter plot of w vs h

In [13]:
fig = px.scatter(
    x=all_w,
    y=all_h,
    labels={"x": "Width", "y": "Height"},
    title="Bounding box size (w vs h)"
)


fig.show()

Historgram of aspect ration between h and w

In [14]:
aspect_ratio = np.array(all_h) / np.array(all_w)

fig = px.histogram(
    x=aspect_ratio,
    title="Aspect ratio between h and w",
    labels={"x": "Height/Width", "y": "count"},
)

fig.update_layout(bargap=0.1)

fig.show()

heatmap of cx and cy

In [15]:
fig = px.density_heatmap(
    x=all_cx,
    y=all_cy,
    title="Heatmap of bounding box center point (cx vs cy)",
    labels={"x": "cx", "y": "cy"}
)
fig.show()

Histogram of relativ size w*h

In [16]:
relative_size = np.array(all_w) * np.array(all_h)

fig = px.histogram(
    x=relative_size,
    title="Relative size of boxes (w×h)",
    labels={"x": "Width × Height", "y": "Number of images"}
)

fig.update_layout(bargap=0.1)

fig.show()

Calculating average brightness per image and checking that the resolution is the same for every image

In [17]:
image_paths_brightness = []
widths, heights, brightness = [], [], []
weights = np.array([0.299, 0.587, 0.114])

for split in ["train", "valid", "test"]:
    image_dir = dataset_path / split / "images"
    images = list(image_dir.glob("*.PNG"))
    
    for image_path in images:
        with Image.open(image_path) as image:
            image_paths_brightness.append(image_path)
            widths.append(image.width)
            heights.append(image.height)
            image_array = np.array(image)
            brightness.append(np.mean(image_array @ weights))
            
check_widths = len(set(widths)) == 1
check_heights = len(set(heights)) == 1

if check_widths and check_heights:
    print(f"All images have the same resolution: {widths[0]}x{heights[0]}")
elif check_widths:
    print(f"All images have the same width: {widths[0]}px, but heights vary")
elif check_heights:
    print(f"All images have the same height: {heights[0]}px, but widths vary")
else:
    print("Images have varying resolutions")

    

All images have the same resolution: 1920x1208


Histogram of average brightness

In [18]:
fig = px.histogram(
    x=brightness,
    title="Average brightness per image (0-255)",
    labels={"x": "Brightness", "y": "Number of images"}
)

fig.update_layout(bargap=0.1)

fig.show()

Plotting the 5 darkest images, 5 images with the smallest relative size boxes.